# Improving on the paper-split baseline: LightGBM and a stronger E-GraphSAGE-M

Notebook A reproduced the setup of the GTCN-G paper (5:2:3 split, E-GraphSAGE-M baseline) and reached a test weighted F1 of 0.9836 and macro F1 of 0.581. This notebook uses the same split and asks what improves on that, with every choice made on the validation set and the test set used once.

| Condition | What changes |
|---|---|
| LightGBM, plain | tabular baseline, no class weights |
| LightGBM, class weights | balanced class weights |
| LightGBM, two-stage, class weights | attack-vs-normal first, then attack type among attacks |
| LightGBM, class weights + tuned class scales | per-class probability multipliers tuned on validation macro F1 |
| GNN, graph | E-GraphSAGE-M of notebook A with square-root class weights and the flow's own features added to the classifier |
| GNN, no graph | the same network where every flow is isolated, so the graph cannot help |

Reported: test weighted F1 (the paper's metric) and macro F1, three seeds, bootstrap intervals over the test set, and paired differences for the questions we care about.

In [ ]:
import copy

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

DEBUG = False
SEEDS = [42] if DEBUG else [42, 43, 44]
PATH = "/kaggle/input/datasets/shahiismyname/unsw-nb15-preprocessed-dataset"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

HIDDEN = 128
DROPOUT = 0.2
LR = 0.001
BATCH = 500
FANOUT = 8
MAX_EPOCHS = 12 if DEBUG else 100
PATIENCE = 12 if DEBUG else 20
BOOTSTRAPS = 100 if DEBUG else 1000
PAPER = {"E-GraphSAGE": 0.8756, "E-GraphSAGE-M": 0.8934, "GAT": 0.9178, "GTCN-G": 0.9512}
NOTEBOOK_A = {"weighted F1": 0.9836, "macro F1": 0.581}

## 1. Data and 5:2:3 split (identical to notebook A)

In [ ]:
df = pd.concat([pd.read_parquet(f"{PATH}/train.parquet"), pd.read_parquet(f"{PATH}/test.parquet")])
graph = pd.concat([pd.read_csv(f"{PATH}/train_graph.csv", index_col="row_id"),
                   pd.read_csv(f"{PATH}/test_graph.csv", index_col="row_id")]).loc[df.index]
if DEBUG:
    df = df.sample(200_000, random_state=0)
    graph = graph.loc[df.index]

class_names = list(joblib.load(f"{PATH}/preprocessing.pkl")["label_encoder"].classes_)
n_classes = len(class_names)
normal = class_names.index("Normal")
attack_ids = [i for i in range(n_classes) if i != normal]
remap = {orig: new for new, orig in enumerate(attack_ids)}
feature_cols = [c for c in df.columns if c != "label"]

train_ids, rest_ids = train_test_split(df.index, train_size=0.5, stratify=df["label"], random_state=0)
val_ids, test_ids = train_test_split(rest_ids, train_size=0.4, stratify=df.loc[rest_ids, "label"], random_state=0)
ids = {"train": train_ids, "val": val_ids, "test": test_ids}
X = {k: df.loc[v, feature_cols] for k, v in ids.items()}
y = {k: df.loc[v, "label"] for k, v in ids.items()}
y_test = y["test"].to_numpy()
print({k: len(v) for k, v in ids.items()}, "| features", len(feature_cols))

results, class_rows, preds = [], [], {}


def scores(y_true, pred):
    per_class = f1_score(y_true, pred, labels=np.arange(n_classes), average=None, zero_division=0)
    return {"weighted F1": f1_score(y_true, pred, average="weighted", zero_division=0),
            "macro F1": per_class.mean(), "accuracy": accuracy_score(y_true, pred)}, per_class


def record(name, seed, pred):
    s, per_class = scores(y_test, pred)
    results.append({"condition": name, "seed": seed, **s})
    class_rows.append({"condition": name, "seed": seed, **dict(zip(class_names, per_class))})
    preds[f"{name}__{seed}"] = np.asarray(pred).astype("int8")
    print(f"{name:<48} seed {seed} | test weighted F1 {s['weighted F1']:.4f} | macro F1 {s['macro F1']:.4f}", flush=True)


def save():
    pd.DataFrame(results).to_csv("/kaggle/working/improved_runs.csv", index=False)
    pd.DataFrame(class_rows).to_csv("/kaggle/working/improved_per_class.csv", index=False)

## 2. LightGBM

Early stopping on validation log loss. The number of leaves is chosen on validation macro F1 with one seed, then used for every LightGBM condition.

In [ ]:
STOP = [lgb.early_stopping(20), lgb.log_evaluation(0)]


def lgbm(seed, weighted, leaves, **kwargs):
    return lgb.LGBMClassifier(class_weight="balanced" if weighted else None, learning_rate=0.08, num_leaves=leaves,
                              n_estimators=500, random_state=seed, verbose=-1, **kwargs)


def single_stage(seed, weighted, leaves):
    model = lgbm(seed, weighted, leaves, objective="multiclass")
    return model.fit(X["train"], y["train"], eval_set=[(X["val"], y["val"])], callbacks=STOP)


def two_stage(seed, leaves):
    is_attack = {k: (v != normal).astype(int) for k, v in y.items()}
    stage1 = lgbm(seed, True, leaves, objective="binary")
    stage1.fit(X["train"], is_attack["train"], eval_set=[(X["val"], is_attack["val"])], callbacks=STOP)
    tr, va = is_attack["train"] == 1, is_attack["val"] == 1
    stage2 = lgbm(seed, True, leaves, objective="multiclass")
    stage2.fit(X["train"][tr], y["train"][tr].map(remap), eval_set=[(X["val"][va], y["val"][va].map(remap))], callbacks=STOP)
    pred = np.full(len(X["test"]), normal)
    flagged = np.where(stage1.predict(X["test"]) == 1)[0]
    pred[flagged] = np.array(attack_ids)[stage2.predict(X["test"].iloc[flagged])]
    return pred


def tune_scales(proba, y_true, passes=2, grid=np.geomspace(0.25, 4, 13)):
    scale = np.ones(n_classes)
    for _ in range(passes):
        for c in range(n_classes):
            trial = []
            for m in grid:
                s = scale.copy()
                s[c] = m
                trial.append(f1_score(y_true, (proba * s).argmax(1), average="macro", zero_division=0))
            scale[c] = grid[int(np.argmax(trial))]
    return scale


search = {}
for leaves in [31, 63, 127, 255]:
    model = single_stage(42, True, leaves)
    search[leaves] = f1_score(y["val"], model.predict(X["val"]), average="macro", zero_division=0)
    print(f"leaves {leaves:>3} | trees {model.best_iteration_:>3} | validation macro F1 {search[leaves]:.4f}", flush=True)
LEAVES = max(search, key=search.get)
print("chosen leaves:", LEAVES)

In [ ]:
for seed in SEEDS:
    record("LightGBM, plain", seed, single_stage(seed, False, LEAVES).predict(X["test"]))
    weighted_model = single_stage(seed, True, LEAVES)
    record("LightGBM, class weights", seed, weighted_model.predict(X["test"]))
    record("LightGBM, two-stage, class weights", seed, two_stage(seed, LEAVES))
    scale = tune_scales(weighted_model.predict_proba(X["val"]), y["val"].to_numpy())
    record("LightGBM, class weights + tuned class scales", seed, (weighted_model.predict_proba(X["test"]) * scale).argmax(1))
    print("tuned scales:", dict(zip(class_names, scale.round(2))))
    save()

## 3. E-GraphSAGE-M, improved

Same sampled mini-batch model as notebook A (8 neighbours per hop, batches of 500) with two changes: square-root balanced class weights in the loss, and the flow's own features concatenated to the two endpoint embeddings in the classifier. The no-graph condition gives every flow its own pair of nodes. Training stops on validation macro F1, with the learning rate halved after 5 epochs without an improvement.

In [ ]:
def build_graph(part, isolated):
    feat_df, graph_df = df.loc[ids[part]], graph.loc[ids[part]]
    n = len(feat_df)
    if isolated:
        node_id = np.arange(2 * n)
    else:
        ip_code, _ = pd.factorize(pd.concat([graph_df["srcip"], graph_df["dstip"]]))
        ports = np.concatenate([graph_df["sport"].fillna(-1), graph_df["dsport"].fillna(-1)]).astype("int64") + 1
        node_id, _ = pd.factorize(ip_code.astype("int64") * 70_000 + ports)
    node_id = torch.from_numpy(node_id).to(DEVICE)
    src, dst = node_id[:n], node_id[n:]

    nodes = torch.cat([src, dst])
    other = torch.cat([dst, src])
    flow = torch.cat([torch.arange(n, device=DEVICE)] * 2)
    order = torch.argsort(nodes)
    ptr = torch.zeros(int(nodes.max()) + 2, dtype=torch.long, device=DEVICE)
    ptr[1:] = torch.cumsum(torch.bincount(nodes), 0)
    return {"src": src, "dst": dst, "nbr": other[order], "eflow": flow[order], "ptr": ptr,
            "x": torch.from_numpy(feat_df[feature_cols].to_numpy("float32")).to(DEVICE),
            "y": torch.from_numpy(feat_df["label"].to_numpy("int64")).to(DEVICE)}


GRAPHS = {isolated: {part: build_graph(part, isolated) for part in ids} for isolated in [False, True]}
for isolated, G in GRAPHS.items():
    for part, g in G.items():
        deg = torch.diff(g["ptr"]).float()
        print(f"{'isolated' if isolated else 'real':<9} {part:<6} nodes {len(deg):>9,} | mean degree {deg.mean():.2f} | single-flow nodes {(deg == 1).float().mean():.1%}")


def sample(g, nodes):
    start = g["ptr"][nodes]
    deg = g["ptr"][nodes + 1] - start
    pos = start[:, None] + (torch.rand(len(nodes), FANOUT, device=DEVICE) * deg[:, None]).long()
    return g["nbr"][pos], g["eflow"][pos]


class EGraphSAGEM(nn.Module):
    def __init__(self, n_feat, n_classes):
        super().__init__()
        self.neigh = nn.ModuleList([nn.Linear(1, HIDDEN, bias=False), nn.Linear(HIDDEN, HIDDEN, bias=False)])
        self.edge = nn.ModuleList([nn.Linear(n_feat, HIDDEN), nn.Linear(n_feat, HIDDEN)])
        self.update = nn.ModuleList([nn.Linear(1 + HIDDEN, HIDDEN), nn.Linear(2 * HIDDEN, HIDDEN)])
        self.drop = nn.Dropout(DROPOUT)
        self.out = nn.Linear(2 * HIDDEN + n_feat, n_classes)

    def forward(self, g, idx):
        ends = torch.cat([g["src"][idx], g["dst"][idx]])
        n1, e1 = sample(g, ends)
        hop1 = torch.cat([ends, n1.flatten()])
        _, e2 = sample(g, hop1)

        ones = torch.ones(len(hop1), 1, device=DEVICE)
        msg = self.neigh[0](ones) + self.edge[0](g["x"][e2].mean(1))
        h1 = self.drop(torch.relu(self.update[0](torch.cat([ones, msg], dim=1))))

        h_ends, h_nbr = h1[:len(ends)], h1[len(ends):].view(len(ends), FANOUT, HIDDEN)
        msg = self.neigh[1](h_nbr.mean(1)) + self.edge[1](g["x"][e1].mean(1))
        h2 = torch.relu(self.update[1](torch.cat([h_ends, msg], dim=1)))
        return self.out(torch.cat([h2[:len(idx)], h2[len(idx):], g["x"][idx]], dim=1))


@torch.no_grad()
def evaluate(model, g, loss_fn, batch=5000):
    model.eval()
    logits = torch.cat([model(g, torch.arange(i, min(i + batch, len(g["src"])), device=DEVICE))
                        for i in range(0, len(g["src"]), batch)])
    return loss_fn(logits, g["y"]).item(), logits.argmax(1).cpu().numpy()


def run_gnn(name, seed, isolated):
    torch.manual_seed(seed)
    G = GRAPHS[isolated]
    weights = np.sqrt(compute_class_weight("balanced", classes=np.arange(n_classes), y=y["train"].to_numpy()))
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE))
    model = EGraphSAGEM(len(feature_cols), n_classes).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4)
    n_train, y_val = len(G["train"]["src"]), y["val"].to_numpy()

    history, best, bad = [], {"f1": -1.0}, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        perm = torch.randperm(n_train, device=DEVICE)
        total = torch.zeros((), device=DEVICE)
        for i in range(0, n_train, BATCH):
            idx = perm[i:i + BATCH]
            loss = loss_fn(model(G["train"], idx), G["train"]["y"][idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.detach() * len(idx)

        val_loss, val_pred = evaluate(model, G["val"], loss_fn)
        val, _ = scores(y_val, val_pred)
        history.append({"condition": name, "seed": seed, "epoch": epoch, "train loss": total.item() / n_train,
                        "val loss": val_loss, "val weighted F1": val["weighted F1"], "val macro F1": val["macro F1"]})
        print(f"{name} seed {seed} epoch {epoch:>3} | train loss {history[-1]['train loss']:.4f} | val loss {val_loss:.4f} "
              f"| val weighted F1 {val['weighted F1']:.4f} | val macro F1 {val['macro F1']:.4f}", flush=True)

        sched.step(val["macro F1"])
        if val["macro F1"] > best["f1"]:
            best, bad = {"f1": val["macro F1"], "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best["state"])
    record(name, seed, evaluate(model, G["test"], loss_fn)[1])
    print(f"best epoch {best['epoch']} of {epoch}")
    return pd.DataFrame(history)

In [ ]:
histories = []
for seed in SEEDS:
    for name, isolated in [("GNN, graph", False), ("GNN, no graph", True)]:
        histories.append(run_gnn(name, seed, isolated))
        pd.concat(histories).to_csv("/kaggle/working/improved_history.csv", index=False)
        save()
history = pd.concat(histories)
runs = pd.DataFrame(results)
class_results = pd.DataFrame(class_rows)
pd.DataFrame(preds, index=ids["test"]).to_parquet("/kaggle/working/improved_test_pred.parquet")

## 4. GNN training curves

In [ ]:
def curves(columns, titles, path):
    fig, axes = plt.subplots(1, len(columns), figsize=(7 * len(columns), 4.5))
    for ax, col, title in zip(axes, columns, titles):
        sns.lineplot(data=history, x="epoch", y=col, hue="condition", ax=ax)
        ax.set_title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=130, bbox_inches="tight")
    plt.show()


curves(["train loss", "val loss"], ["Training loss", "Validation loss"], "/kaggle/working/fig_01.png")
curves(["val weighted F1", "val macro F1"], ["Validation weighted F1", "Validation macro F1"], "/kaggle/working/fig_02.png")

## 5. Test results

Paper numbers are as reported in the paper. "Notebook A" is our replication of its E-GraphSAGE-M baseline.

In [ ]:
summary = runs.groupby("condition", sort=False)[["weighted F1", "macro F1", "accuracy"]].agg(["mean", "std"])
print(summary.round(4).to_string())
summary.to_csv("/kaggle/working/improved_summary.csv")

order = list(summary.index)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
weighted = {**{f"paper: {k}": v for k, v in PAPER.items()}, "notebook A: E-GraphSAGE-M": NOTEBOOK_A["weighted F1"],
            **runs.groupby("condition")["weighted F1"].mean().loc[order].to_dict()}
axes[0].barh(list(weighted), list(weighted.values()),
             color=["gray"] * len(PAPER) + ["darkorange"] + ["steelblue"] * len(order))
axes[0].set_xlim(0.85, 1.0)
axes[0].set_title("Test weighted F1 (paper's metric)")
macro = {"notebook A: E-GraphSAGE-M": NOTEBOOK_A["macro F1"], **runs.groupby("condition")["macro F1"].mean().loc[order].to_dict()}
axes[1].barh(list(macro), list(macro.values()), color=["darkorange"] + ["steelblue"] * len(order))
axes[1].set_xlim(0.4, 0.8)
axes[1].set_title("Test macro F1")
for ax in axes:
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig("/kaggle/working/fig_03.png", dpi=130, bbox_inches="tight")
plt.show()

per_class = class_results.groupby("condition", sort=False)[class_names].mean()
plt.figure(figsize=(12, 4.5))
sns.heatmap(per_class, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1)
plt.title("Test F1 per class (mean over seeds)")
plt.ylabel("")
plt.tight_layout()
plt.savefig("/kaggle/working/fig_04.png", dpi=130, bbox_inches="tight")
plt.show()

## 6. Bootstrap intervals and paired differences

Test rows are resampled with replacement (the same rows for every condition), using the predictions of seed 42. Intervals show test-set sampling noise only.

In [ ]:
def f1_from_counts(counts):
    tp = np.diag(counts)
    f1 = 2 * tp / np.maximum(counts.sum(0) + counts.sum(1), 1)
    return (f1 * counts.sum(1)).sum() / counts.sum(), f1.mean()


names = order
pred_test = {n: preds[f"{n}__{SEEDS[0]}"].astype("int64") for n in names}
rng = np.random.default_rng(0)
boot = {n: np.zeros((BOOTSTRAPS, 2)) for n in names}
for b in range(BOOTSTRAPS):
    idx = rng.integers(0, len(y_test), len(y_test))
    for n in names:
        counts = np.bincount(y_test[idx] * n_classes + pred_test[n][idx], minlength=n_classes ** 2).reshape(n_classes, n_classes)
        boot[n][b] = f1_from_counts(counts)

table = pd.DataFrame({n: {"weighted F1": f"{np.mean(boot[n][:, 0]):.4f} ({np.percentile(boot[n][:, 0], 2.5):.4f} to {np.percentile(boot[n][:, 0], 97.5):.4f})",
                          "macro F1": f"{np.mean(boot[n][:, 1]):.4f} ({np.percentile(boot[n][:, 1], 2.5):.4f} to {np.percentile(boot[n][:, 1], 97.5):.4f})"}
                      for n in names}).T
print(table.to_string())

pairs = [("GNN, graph", "GNN, no graph"),
         ("LightGBM, class weights", "LightGBM, plain"),
         ("LightGBM, two-stage, class weights", "LightGBM, class weights"),
         ("LightGBM, class weights + tuned class scales", "LightGBM, class weights"),
         ("LightGBM, class weights", "GNN, graph")]
rows = []
for a, b in pairs:
    for j, metric in enumerate(["weighted F1", "macro F1"]):
        d = boot[a][:, j] - boot[b][:, j]
        rows.append({"A": a, "B": b, "metric": metric, "A minus B": d.mean(),
                     "95% interval": f"{np.percentile(d, 2.5):+.4f} to {np.percentile(d, 97.5):+.4f}"})
paired = pd.DataFrame(rows)
print(paired.round(4).to_string(index=False))
table.to_csv("/kaggle/working/improved_bootstrap.csv")
paired.to_csv("/kaggle/working/improved_paired_differences.csv", index=False)